# ChestX-ray14 V2.1: export the demo model to ONNX

Exports the DenseNet121 (seed 2) model from the V2.1 run to ONNX so it can run in a web browser, checks the ONNX outputs against PyTorch, and saves three example test images with their reference outputs.

Inputs: the output of the notebook *ChestX-ray14 V2.1: Seeds, Calibration, Subgroups* and the dataset `nih-chest-xrays/data`. CPU is enough. Internet = On (to install `onnx` and `onnxruntime`).

The ONNX model takes a normalised 1x3x224x224 image and returns:
- `probs`: raw sigmoid probabilities for 14 labels (calibration is applied in the app)
- `cams`: class activation maps, 14x7x7. For DenseNet121 (global average pooling followed by one linear layer) these are equal to Grad-CAM up to a constant factor, so after normalisation they give the same heatmap.

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import glob, json, os, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models

LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumonia",
    "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
OUT = Path("/kaggle/working/demo_export")
(OUT / "examples").mkdir(parents=True, exist_ok=True)


def find_one(pattern):
    hits = sorted(glob.glob("/kaggle/input/**/" + pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(pattern + " not found. Add the V2.1 notebook output and nih-chest-xrays/data as inputs.")
    return Path(hits[0])


weights = find_one("densenet121_seed2_best.pt")
entry = find_one("Data_Entry_2017.csv")
test_list = find_one("test_list.txt")
print(weights, entry, test_list, sep="\n")

In [ ]:
class DemoModel(nn.Module):
    """DenseNet121 with two outputs: raw probabilities and class activation maps."""

    def __init__(self, net):
        super().__init__()
        self.features = net.features
        self.classifier = net.classifier

    def forward(self, image):
        feats = F.relu(self.features(image))
        logits = self.classifier(F.adaptive_avg_pool2d(feats, 1).flatten(1))
        cams = F.conv2d(feats, self.classifier.weight[:, :, None, None])
        return torch.sigmoid(logits), cams


net = models.densenet121(weights=None)
net.classifier = nn.Linear(net.classifier.in_features, len(LABELS))
net.load_state_dict(torch.load(weights, map_location="cpu", weights_only=True))
model = DemoModel(net).eval()

onnx_path = OUT / "chest_xray_densenet121.onnx"
export_args = dict(input_names=["image"], output_names=["probs", "cams"], opset_version=17,
                   do_constant_folding=True)
try:
    torch.onnx.export(model, torch.zeros(1, 3, 224, 224), str(onnx_path), dynamo=False, **export_args)
except TypeError:  # older PyTorch without the dynamo argument
    torch.onnx.export(model, torch.zeros(1, 3, 224, 224), str(onnx_path), **export_args)
print(onnx_path, round(onnx_path.stat().st_size / 1e6, 1), "MB")

In [ ]:
import onnx
import onnxruntime as ort

onnx.checker.check_model(str(onnx_path))
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)


def preprocess(path):
    gray = np.asarray(Image.open(path).convert("L").resize((224, 224), Image.BILINEAR), dtype=np.float32) / 255.0
    return ((np.repeat(gray[None, None], 3, axis=1) - MEAN) / STD).astype(np.float32)


rng = np.random.default_rng(0)
x = rng.normal(size=(1, 3, 224, 224)).astype(np.float32)
with torch.no_grad():
    p_t, c_t = model(torch.from_numpy(x))
p_o, c_o = session.run(None, {"image": x})
print("max |probs diff|:", float(np.abs(p_t.numpy() - p_o).max()))
print("max |cams diff| / max |cam|:", float(np.abs(c_t.numpy() - c_o).max() / np.abs(c_t.numpy()).max()))
assert np.abs(p_t.numpy() - p_o).max() < 1e-4

In [ ]:
# Three example images from the official test split, chosen by a fixed rule: the first image
# (by file name) whose only label is Cardiomegaly, then Effusion, then No Finding.
df = pd.read_csv(entry)
test_names = set(Path(test_list).read_text().split())
df = df[df["Image Index"].isin(test_names)].sort_values("Image Index")
paths = {}
for root, _, files in os.walk(entry.parent):
    for f in files:
        if f.endswith(".png"):
            paths[f] = os.path.join(root, f)

reference = []
for i, label in enumerate(["Cardiomegaly", "Effusion", "No Finding"], start=1):
    row = df[df["Finding Labels"] == label].iloc[0]
    out = OUT / "examples" / f"example_{i}.png"
    Image.open(paths[row["Image Index"]]).convert("L").resize((512, 512), Image.BILINEAR).save(out)
    probs, cams = session.run(None, {"image": preprocess(out)})
    reference.append({"file": out.name, "source_image": row["Image Index"], "labels": label,
                      "age": int(row["Patient Age"]), "sex": row["Patient Gender"],
                      "raw_probs": {l: round(float(v), 6) for l, v in zip(LABELS, probs[0])}})
    print(out.name, row["Image Index"], label)
(OUT / "reference_outputs.json").write_text(json.dumps(reference, indent=2))
shutil.copy(entry, OUT / "Data_Entry_2017.csv")

In [ ]:
for f in sorted(OUT.rglob("*")):
    if f.is_file():
        print(f.relative_to(OUT), round(f.stat().st_size / 1e6, 2), "MB")